## 0. Fast resume (run this and only this on every reconnect)

Combines cell 2 (mount + clone + env) and cell 8 (parquet mirror) into
one block. After running this, the runtime is fully set up and the
parquet cache is on local SSD ready for training. Skip cells 1-8 below;
go straight to whatever section you want (M1 train is section 5d).

First time ever (no Drive cache yet): this falls back to building the
cache, ~25 min. Every reconnect after: ~3 min.


In [ ]:
# === Fast resume — every reconnect runs this and nothing else from setup ===
import os, subprocess, sys, pathlib, time, shutil
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

REPO_URL      = 'https://github.com/nikku03/cell.git'
BRANCH        = 'claude/vectorize-gex-propensity-NRqBW'
REPO          = pathlib.Path('/content/cell')
DATASET       = pathlib.Path('/content/drive/MyDrive/Luthey-Schulten-Lab-Minimal_Cell-db048ac')
PARQUET_DRIVE = pathlib.Path('/content/drive/MyDrive/cell_parquet_cache')
PARQUET_LOCAL = pathlib.Path('/content/parquet_cache')
CKPT_DIR      = pathlib.Path('/content/drive/MyDrive/cell_count_dynamics')

if not REPO.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(REPO)])
subprocess.check_call(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH])
subprocess.check_call(['git', '-C', str(REPO), 'checkout', BRANCH])
subprocess.check_call(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'pyarrow', 'matplotlib'])

sys.path.insert(0, str(REPO / 'cell_sim'))
sys.path.insert(0, str(REPO))
os.environ['LSDATA_ROOT']          = str(DATASET)
os.environ['LSDATA_PARQUET_CACHE'] = str(PARQUET_LOCAL)
# Reduce CUDA allocator fragmentation — relevant when allocating
# large per-edge activation tensors that can leave gaps after free.
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
for d in (PARQUET_DRIVE, PARQUET_LOCAL, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

from cell_sim.data import lsdata
n_species = len(lsdata.replicate_species(1))

# Mirror Drive → local SSD (skips files already mirrored)
n_drive = len(list(PARQUET_DRIVE.glob('*.parquet')))
if n_drive == 0:
    print('Drive cache empty — running full build (one-time, ~25 min)...')
    for i in range(1, 51):
        out = PARQUET_DRIVE / f'counts_and_fluxes.{i}.parquet'
        if out.exists(): continue
        df = lsdata.load_replicate(i)
        df.to_parquet(out, compression='zstd')
        print(f'  built rep {i}')

t0 = time.time()
n_copied = 0
for src in sorted(PARQUET_DRIVE.glob('*.parquet')):
    dst = PARQUET_LOCAL / src.name
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        continue
    shutil.copyfile(src, dst); n_copied += 1

import torch
print(f'\ntorch       : {torch.__version__}  cuda={torch.cuda.is_available()}')
print(f'repo HEAD   : {subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"]).decode().strip()}')
print(f'parquet drive: {len(list(PARQUET_DRIVE.glob("*.parquet")))} files')
print(f'parquet local: {len(list(PARQUET_LOCAL.glob("*.parquet")))} files'
      f'  (mirrored {n_copied} this session in {time.time()-t0:.0f}s)')
print(f'n_species   : {n_species}')
print('\nReady. Skip to section 5c (gates) or 5d (M1 train).')


# Train the count-dynamics surrogate on the Luthey-Schulten Minimal Cell trajectories

**What this notebook does**

1. Mounts your Drive copy of `Luthey-Schulten-Lab-Minimal_Cell-db048ac/`.
2. Pulls the `claude/vectorize-gex-propensity-NRqBW` branch of `nikku03/cell`.
3. Probes the 8572 rows of `counts_and_fluxes` to see how many are species counts vs reaction fluxes.
4. One-time converts the 50 replicates from tar+CSV into local parquet so per-epoch I/O drops from minutes to seconds.
5. Trains the count-dynamics MLP defined in `cell_sim/layer_ml/count_dynamics.py` (~18.6 M params, predicts Δlog1p of the 8572-vector).
6. Inspects per-species R² on the held-out replicate and plots a few trajectories.

**Runtime expectations** (on a Colab T4):
- Cells 1–3 (setup + probe): under a minute.
- Cell 4 (parquet conversion): ≈20–40 minutes the first time, near-zero on later runs.
- Cell 5 (training): ≈10 minutes/epoch with parquet cache; tens of minutes/epoch without it.
- Cell 6 (inspection): under a minute.

If anything goes sideways the notebook is safe to re-run from any cell.

## 1. Mount Drive + clone the repo + install requirements

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, subprocess, sys, pathlib

REPO_URL      = 'https://github.com/nikku03/cell.git'
BRANCH        = 'claude/vectorize-gex-propensity-NRqBW'
REPO          = pathlib.Path('/content/cell')
DATASET       = pathlib.Path('/content/drive/MyDrive/Luthey-Schulten-Lab-Minimal_Cell-db048ac')
# Two-tier parquet cache: persistent on Drive, fast mirror on local SSD.
# First-time conversion writes to Drive (one-time ~25 min). Every reconnect
# rsyncs Drive → local (~3 min). Training reads from local at SSD speed.
PARQUET_DRIVE = pathlib.Path('/content/drive/MyDrive/cell_parquet_cache')
PARQUET_LOCAL = pathlib.Path('/content/parquet_cache')
CKPT_DIR      = pathlib.Path('/content/drive/MyDrive/cell_count_dynamics')

if not REPO.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(REPO)])
subprocess.check_call(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH])
subprocess.check_call(['git', '-C', str(REPO), 'checkout', BRANCH])
subprocess.check_call(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'pyarrow', 'matplotlib'])

sys.path.insert(0, str(REPO / 'cell_sim'))
sys.path.insert(0, str(REPO))
os.environ['LSDATA_ROOT']          = str(DATASET)
os.environ['LSDATA_PARQUET_CACHE'] = str(PARQUET_LOCAL)   # reads go here
PARQUET_DRIVE.mkdir(parents=True, exist_ok=True)
PARQUET_LOCAL.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

import torch
print(f'torch         : {torch.__version__}  cuda={torch.cuda.is_available()}')
print(f'repo branch   : {subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "--abbrev-ref", "HEAD"]).decode().strip()}')
print(f'repo HEAD     : {subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"]).decode().strip()}')
print(f'dataset       : {DATASET}  exists={DATASET.exists()}')
print(f'parquet drive : {PARQUET_DRIVE}  '
      f'({len(list(PARQUET_DRIVE.glob("*.parquet")))} files)')
print(f'parquet local : {PARQUET_LOCAL}  '
      f'({len(list(PARQUET_LOCAL.glob("*.parquet")))} files)')
print(f'checkpoints   : {CKPT_DIR}')


## 2. Sanity-check the registry + count rows

In [ ]:
from cell_sim.data import lsdata

info = lsdata.check_registry()
print(f'root       : {info["root"]}')
print(f'present    : {len(info["present"])}/27 small assets')
if info['missing']:
    print(f'missing    : {info["missing"]}')

n_species = len(lsdata.replicate_species(1))
rep_indices = lsdata.list_replicates()
print(f'\nreplicate count : {len(rep_indices)} (indices {min(rep_indices)}..{max(rep_indices)})')
print(f'rows / replicate: {n_species}')

## 3. Probe the row composition of `counts_and_fluxes`

We want to know how many of the 8572 rows are species counts (non-negative integers) vs reaction fluxes (signed floats, sometimes NaN). The training transform handles all three cases now, but if fluxes dominate the row count it's worth knowing for interpretation.

In [ ]:
import collections
import numpy as np

species_names = lsdata.replicate_species(1)
prefixes = collections.Counter(s.split('_')[0] for s in species_names)
print('Top 15 row prefixes:')
for p, n in prefixes.most_common(15):
    print(f'  {p:>20s}  {n:>5d}')

df_t100 = lsdata.load_replicate(1, time_start=100.0, time_end=100.0)
vals = df_t100.iloc[:, 0].to_numpy()
print(f'\nAt t=100s in replicate 1:')
print(f'  finite      : {np.isfinite(vals).sum():>5d}')
print(f'  NaN         : {np.isnan(vals).sum():>5d}')
print(f'  negative    : {(vals < 0).sum():>5d}')
print(f'  zero        : {(vals == 0).sum():>5d}')
print(f'  positive    : {(vals > 0).sum():>5d}')
print(f'  min / max   : {np.nanmin(vals):.3f} / {np.nanmax(vals):.3f}')

## 4. Two-tier parquet cache: Drive (persistent) + local SSD (fast)

Reading 49 replicates × 250 MB CSVs from a tarball on Drive is ~25 min
per epoch. Parquet (zstd) shrinks each replicate to ~80 MB and reads
100× faster than CSV.

Two tiers because Colab's local SSD is wiped on every disconnect:
- **Drive cache** (`PARQUET_DRIVE`): persistent. First-time-ever run
  takes ~25 min to build all 50 files; idempotent re-runs are instant.
- **Local SSD mirror** (`PARQUET_LOCAL`): fast. Every reconnect, copy
  Drive → local (~3 min for ~4 GB). Training reads from here.

lsdata.load_replicate() respects `LSDATA_PARQUET_CACHE` (set to local)
and falls back to streaming the tar if the parquet isn't there.


In [ ]:
import time, shutil

# Step 1: build parquet cache on Drive (one-time, ~25 min the first run;
# instant if already done). The if-exists check makes this idempotent.
print('=== Step 1: ensure parquet cache on Drive ===')
t0 = time.time()
n_built = 0
for i in range(1, 51):
    out = PARQUET_DRIVE / f'counts_and_fluxes.{i}.parquet'
    if out.exists():
        continue
    t = time.time()
    df = lsdata.load_replicate(i)               # tar stream from Drive
    df.to_parquet(out, compression='zstd')
    print(f'  rep {i:>2d} → drive  {out.stat().st_size/1024**2:>6.1f} MB  '
          f'(read+write {time.time()-t:.1f}s)')
    n_built += 1
drive_total = sum(p.stat().st_size for p in PARQUET_DRIVE.glob('*.parquet'))
print(f'\nDrive cache: {drive_total/1024**3:.2f} GB across '
      f'{len(list(PARQUET_DRIVE.glob("*.parquet")))} files'
      f'  (built {n_built} new, total step time {time.time()-t0:.0f}s)')

# Step 2: mirror Drive → local SSD for fast reads during training. Only
# copies missing files; if local already has them (warm runtime), skip.
print('\n=== Step 2: mirror Drive cache → local SSD ===')
t0 = time.time()
n_copied = 0
for src in sorted(PARQUET_DRIVE.glob('*.parquet')):
    dst = PARQUET_LOCAL / src.name
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        continue
    shutil.copyfile(src, dst)
    n_copied += 1
print(f'mirrored {n_copied} files to {PARQUET_LOCAL} in {time.time()-t0:.0f}s')
local_total = sum(p.stat().st_size for p in PARQUET_LOCAL.glob('*.parquet'))
print(f'Local cache: {local_total/1024**3:.2f} GB across '
      f'{len(list(PARQUET_LOCAL.glob("*.parquet")))} files'
      f'  — training will read from here at SSD speed')


## 5. Train the count-dynamics surrogate

Defaults: 49 replicates train, 1 hold-out, 2 epochs, batch=256, hidden=1024, 2 residual blocks. ~18.6 M params. Loss is MSE on Δ (signed-log1p of the 8572-vector).

Bump `n_epochs` once you see a healthy first-epoch loss curve.

In [ ]:
from cell_sim.lgnn.training.train_mlp import (
    TrainConfig, train_count_dynamics)

cfg = TrainConfig(
    n_species=n_species,
    hidden=1024, n_blocks=2, dropout=0.0,
    batch_size=256, lr=3e-4, weight_decay=1e-5,
    train_replicates=tuple(range(1, 50)),
    val_replicates=(50,),
    n_epochs=2,
    device='auto',
    log_every=200,
)
ckpt_path = CKPT_DIR / 'count_dynamics_v0.pt'
out = train_count_dynamics(cfg, lsdata, checkpoint_path=ckpt_path)
print('\n=== Training complete ===')
print(f'best val MSE : {out["best_val_mse"]:.4f}')
print(f'history      : {out["history"]}')
print(f'checkpoint   : {ckpt_path}')

## 5b. Build the species reaction graph (week-2 prep)

Parses the iMB155 COBRA JSON and builds a species×species edge index where two rows are connected if they appear in the same reaction. Rows in the 8572-vector that don't match an SBML metabolite ID get a self-loop only — the M1 GNN will fall back to a per-node MLP on those, identical to the M0 MLP behaviour. This cell only needs to run once; it saves the graph to Drive so future training cells can `load_species_graph(...)`.

In [ ]:
from cell_sim.lgnn.data.species_graph import (
    build_species_graph, diagnose_row_matching,
    graph_summary, save_species_graph,
    simulator_edge_summary, flux_edge_summary)

row_names = lsdata.replicate_species(1)

# 1. SBML diagnostic — choose best metabolite-graph source
candidates = []
syn3a_xml = REPO / 'cell_sim' / 'data' / 'Minimal_Cell_ComplexFormation' / 'input_data' / 'Syn3A_updated.xml'
if syn3a_xml.exists():
    candidates.append(('Syn3A_updated.xml', syn3a_xml))
try:
    candidates.append(('iMB155 COBRA JSON', lsdata.get('cobra_imb155')))
except Exception as e:
    print(f'iMB155 not available ({e}), skipping')

best = None
for label, p in candidates:
    print(f'\n=== sbml diagnostic: {label} ===')
    rep = diagnose_row_matching(row_names, p)
    for k in ['n_sbml_species', 'n_total_match', 'matches_per_rule']:
        print(f'  {k:25s}  {rep[k]}')
    if best is None or rep['n_total_match'] > best[1]['n_total_match']:
        best = (label, rep, p)
label, rep, src_path = best

# 2. Simulator-edge diagnostic — central-dogma chain
print(f'\n=== simulator-edge diagnostic ===')
sim_rep = simulator_edge_summary(row_names)
print(f'  n_loci_seen           : {sim_rep["n_loci_seen"]}')
print(f'  n_loci_with_G_R_P_all : {sim_rep["n_loci_with_G_R_P_all"]}')
print(f'  total_simulator_edges : {sim_rep["total_simulator_edges"]}')
print(f'  edges per pattern:')
for label_, n in sim_rep['edges_per_pattern'].items():
    print(f'    {label_:50s}  {n}')
print(f'  sample loci           : {sim_rep["sample_loci"]}')

# 3. Flux-edge diagnostic — F_<rxn_id> coverage
print(f'\n=== flux-edge diagnostic ===')
flux_rep = flux_edge_summary(row_names, src_path)
for k in ('n_sbml_reactions', 'n_with_F_avg_row', 'n_with_F_end_row',
         'n_reactions_covered', 'n_unmatched_reactions'):
    print(f'  {k:25s}  {flux_rep[k]}')
print(f'  sample (rxn, has_avg, has_end):')
for s_ in flux_rep['sample_with_flux_rows']:
    print(f'    {s_}')

# 4. Build the full graph (SBML + simulator + flux + self-loops)
print(f'\nBuilding graph with: {label} (matched {rep["n_total_match"]} rows)\n'
      f'  + simulator edges + flux↔species edges')
g = build_species_graph(row_names, src_path, include_simulator_edges=True)
print('\nGraph stats:')
for k, v in graph_summary(g).items():
    print(f'  {k:30s}  {v}')

graph_path = CKPT_DIR / 'species_graph_full.pt'
save_species_graph(g, graph_path)
print(f'\nsaved -> {graph_path}')


## 5c. Pre-M1 acceptance gate

Before spending a T4-hour on M1 training, the graph and the data loader
have to clear four checks. Each line below evaluates to PASS or FAIL —
if any fails, fix that thing before training M1.

| # | criterion | rationale |
|---|---|---|
| 1 | `n_orphan_nodes` ≤ 4500 (without flux fix) or ≤ 2500 (with flux fix) | the message-passing layers must touch the rows we care about |
| 2 | `n_loci_with_G_R_P_all` ≥ 450 | central-dogma parser caught nearly all loci |
| 3 | every PM_xxxx row has a non-empty incoming edge list from its RP_xxxx | the chain is wired correctly for the M1 thesis rows |
| 4 | M0 with `buffered_shuffle=True, buffer_size=8` is within ±0.05 of the M0a top-100 median R² | loader change doesn't break the baseline before we re-use it for M1 |


In [ ]:
# === Gate 1, 2, 3 — graph-side checks ===
from cell_sim.lgnn.data.species_graph import load_species_graph, EdgeKind
import re

g = load_species_graph(graph_path)
stats = graph_summary(g)
n_orph = stats['n_orphan_nodes']
sim_rep = simulator_edge_summary(g.row_names)
n_loci_full = sim_rep['n_loci_with_G_R_P_all']

# (1) orphan-node budget. Threshold depends on whether flux edges fired.
n_flux = stats['edges_per_kind']['FLUX_COUPLING']
thresh1 = 2500 if n_flux > 0 else 4500
g1 = (n_orph <= thresh1)
print(f'[{"PASS" if g1 else "FAIL"}] gate 1   n_orphan_nodes={n_orph}'
      f'  (threshold {thresh1})  flux_edges={n_flux}')

# (2) locus-coverage budget
g2 = (n_loci_full >= 450)
print(f'[{"PASS" if g2 else "FAIL"}] gate 2   n_loci_with_G_R_P_all={n_loci_full}'
      f'  (threshold 450)')

# (3) PM_xxxx rows must each have ≥1 incoming non-self edge from their
# IMMEDIATE chain predecessors. Per rxns_CME.py the chain is
# RP_<l> ↔ R_<l> ↔ RPM_<l> ↔ PM_<l>  with  P_<l> ↔ PM_<l>  side branch,
# so PM_<l>'s upstream is RPM_<l> and/or P_<l> (not RP_<l>, which is
# 3 hops away). The earlier 'incoming from RP' check was wrong-headed.
name_to_idx = {n: i for i, n in enumerate(g.row_names)}
src, dst = g.edge_index[0].tolist(), g.edge_index[1].tolist()
kind = g.edge_kind.tolist()
pm_rows = [n for n in g.row_names if re.match(r'^PM_\d{3,}', n)]
pm_set = set(pm_rows)
incoming_chain = {n: 0 for n in pm_rows}
for s_, d_, k_ in zip(src, dst, kind):
    if k_ == int(EdgeKind.SELF_LOOP):
        continue
    dst_name = g.row_names[d_]
    if dst_name not in pm_set:
        continue
    src_name = g.row_names[s_]
    locus = dst_name.split('_', 1)[1].split('_', 1)[0]
    if src_name == f'RPM_{locus}' or src_name == f'P_{locus}':
        incoming_chain[dst_name] += 1
n_pm = len(pm_rows)
n_pm_connected = sum(1 for v in incoming_chain.values() if v > 0)
g3 = (n_pm_connected >= 0.95 * n_pm) if n_pm > 0 else True
print(f'[{"PASS" if g3 else "FAIL"}] gate 3   PM_xxxx with chain-upstream'
      f' (RPM_<l> or P_<l>) incoming edges: {n_pm_connected}/{n_pm}')
if not g3:
    sample_orphan_pm = [n for n, v in incoming_chain.items() if v == 0][:5]
    print(f'           sample orphan PM_xxxx (no RPM_/P_ → PM_ edge): {sample_orphan_pm}')

print('\n--- gate 4 (loader-change M0 sanity) is a separate train run; '
      'see next cell ---')


In [ ]:
# === Gate 4 — re-run M0 with buffer_size=8, compare to M0a baseline ===
# Skip this cell if you already have the gate-4 measurement; it costs
# the same as one M0 training run (~45 min on T4).
from cell_sim.lgnn.training.train_mlp import TrainConfig, train_count_dynamics
from cell_sim.lgnn.data.dataset import replicate_to_log1p_array
import numpy as np, torch
from cell_sim.lgnn.models.mlp_baseline import CountDynamicsMLP

cfg_g4 = TrainConfig(
    n_species=n_species, hidden=1024, n_blocks=2, dropout=0.0,
    batch_size=256, lr=3e-4, weight_decay=1e-5,
    train_replicates=tuple(range(1, 50)), val_replicates=(50,),
    n_epochs=2, device='auto', log_every=200,
    buffered_shuffle=True, buffer_size=8,
)
ckpt_g4 = CKPT_DIR / 'count_dynamics_v0_buf8.pt'
_ = train_count_dynamics(cfg_g4, lsdata, checkpoint_path=ckpt_g4)

# Compute top-100 high-variance median R² on rep 50
model_g4 = CountDynamicsMLP(n_species=cfg_g4.n_species,
                            hidden=cfg_g4.hidden, n_blocks=cfg_g4.n_blocks)
model_g4.load_state_dict(torch.load(ckpt_g4, weights_only=False)['state_dict'])
model_g4.eval()
df50 = lsdata.load_replicate(50)
X = replicate_to_log1p_array(df50)
DX = X[1:] - X[:-1]
X_in = X[:-1]                      # slice FIRST so P has T-1 rows
with torch.no_grad():
    P = []
    for s in range(0, X_in.shape[0], 512):
        P.append(model_g4(torch.from_numpy(X_in[s:s+512])).numpy())
P = np.concatenate(P, axis=0)
assert P.shape == DX.shape, (P.shape, DX.shape)
ss_res = ((P - DX) ** 2).sum(axis=0)
ss_tot = ((DX - DX.mean(axis=0, keepdims=True)) ** 2).sum(axis=0)
r2_g4 = np.where(ss_tot > 0, 1 - ss_res / np.where(ss_tot > 0, ss_tot, 1), np.nan)
top100 = np.argsort(X.var(axis=0))[::-1][:100]
median_g4 = float(np.nanmedian(r2_g4[top100]))

M0A_MEDIAN = -0.502         # see EXPERIMENTS.md, M0a row
delta = median_g4 - M0A_MEDIAN
g4 = abs(delta) <= 0.05
print(f'\n[{"PASS" if g4 else "FAIL"}] gate 4   M0(buf=8) top-100 median R² = {median_g4:+.3f}'
      f'  (M0a baseline {M0A_MEDIAN:+.3f}, delta {delta:+.3f})')
if g4:
    print('         → safe to use buffered_shuffle=True, buffer_size=8 for M1')
else:
    print('         → loader change degraded M0; investigate before training M1')


## 5d. Train M1 — hetero-edge GNN delta-predictor

First M1 run uses the same single-step Δsigned-log1p MSE loss as M0
(so headline A is directly comparable), `hidden=64`, `n_layers=3`,
gradient checkpointing on (k=1 doesn't strictly need it but exercises
the path), sequential loader (gate 4 said this is fine for M0; open
question for M1, addressed by the M1.b ablation later).

Headline targets:
- A (top-100 high-var median R²) ≥ −0.50 (matches M0a)
- B (PM_xxxx median R²) ≥ −1.0 (graph thesis crosses recovery threshold)


In [ ]:
from cell_sim.lgnn.training.train_gnn import GNNTrainConfig, train_gnn
from cell_sim.lgnn.data.species_graph import load_species_graph

g_full = load_species_graph(graph_path)
assert g_full.n_nodes == n_species, (g_full.n_nodes, n_species)

cfg_m1 = GNNTrainConfig(
    n_species=n_species,
    hidden=64, n_layers=3,
    batch_size=256, lr=3e-4, weight_decay=1e-5,
    train_replicates=tuple(range(1, 50)),
    val_replicates=(50,),
    n_epochs=2,
    device='auto', seed=42, log_every=200,
    use_checkpoint=True,
    buffered_shuffle=False,        # match M0a; gate 4 informational
)
ckpt_m1 = CKPT_DIR / 'count_dynamics_gnn_v1.pt'
out_m1 = train_gnn(cfg_m1, lsdata, g_full, checkpoint_path=ckpt_m1)
print('\n=== M1 training complete ===')
print(f'best val MSE : {out_m1["best_val_mse"]:.4f}')
print(f'history      : {out_m1["history"]}')
print(f'checkpoint   : {ckpt_m1}')


In [ ]:
# === M1 headline metrics on rep 50, mirroring the M0 eval cell ===
import numpy as np, re, torch
from cell_sim.lgnn.models.gnn_v1 import CellGNNv1
from cell_sim.lgnn.data.dataset import replicate_to_log1p_array

ckpt = torch.load(ckpt_m1, map_location='cpu', weights_only=False)
model_m1 = CellGNNv1(graph=g_full, hidden=cfg_m1.hidden,
                     n_layers=cfg_m1.n_layers, use_checkpoint=False)
model_m1.load_state_dict(ckpt['state_dict'])
model_m1.eval()

df = lsdata.load_replicate(50)
X = replicate_to_log1p_array(df)
DX = X[1:] - X[:-1]
X_in = X[:-1]
with torch.no_grad():
    P = []
    for s in range(0, X_in.shape[0], 256):
        P.append(model_m1(torch.from_numpy(X_in[s:s+256])).numpy())
PRED_M1 = np.concatenate(P, axis=0)
assert PRED_M1.shape == DX.shape

ss_res = ((PRED_M1 - DX) ** 2).sum(axis=0)
ss_tot = ((DX - DX.mean(axis=0, keepdims=True)) ** 2).sum(axis=0)
with np.errstate(divide='ignore', invalid='ignore'):
    r2_m1 = 1.0 - ss_res / np.where(ss_tot > 0, ss_tot, 1.0)
r2_m1 = np.where(ss_tot > 0, r2_m1, np.nan)

names = list(df.index)
var_per_row = X.var(axis=0)

# Headline A — top-100 high-variance
top_idx = np.argsort(var_per_row)[::-1][:100]
print('=== M1 headline A: top-100 high-variance rows ===')
print(f'  median R² : {np.nanmedian(r2_m1[top_idx]):+.3f}'
      f'   (M0a baseline {-0.502:+.3f})')
print(f'  mean R²   : {np.nanmean(r2_m1[top_idx]):+.3f}')
print(f'  >0.5      : {(r2_m1[top_idx] > 0.5).sum()} / 100')
print(f'  >0.0      : {(r2_m1[top_idx] > 0.0).sum()} / 100')

# Headline B — PM_xxxx (the M1 thesis test)
pm_idx = np.array([i for i, n in enumerate(names) if re.match(r'^PM_\d{3,}', n)],
                  dtype=np.int64)
if len(pm_idx) > 0:
    pm_r2 = r2_m1[pm_idx]
    pm_finite = pm_r2[np.isfinite(pm_r2)]
    print(f'\n=== M1 headline B: PM_xxxx rows (the thesis test) ===')
    print(f'  count       : {len(pm_idx)} ({np.isfinite(pm_r2).sum()} finite)')
    print(f'  median R²   : {np.nanmedian(pm_r2):+.3f}'
          f'   (M0 baseline {-1.494:+.3f})')
    print(f'  10th pct R² : {np.nanpercentile(pm_r2, 10):+.3f}'
          f'   (M0 baseline {-4.572:+.3f})')
    print(f'  >0.0        : {(pm_finite > 0).sum()} / {len(pm_finite)}'
          f'   (M0: 0 / 453)')
    print(f'  > -1.0      : {(pm_finite > -1.0).sum()} / {len(pm_finite)}'
          f'   (M0: 140 / 453)')

    # Verdict
    pm_median = float(np.nanmedian(pm_r2))
    if pm_median >= 0.0:
        print('\n*** GRAPH THESIS WIN: PM median R² ≥ 0 ***')
    elif pm_median >= -1.0:
        print('\n*** GRAPH THESIS RECOVERY: PM median R² ≥ -1.0 ***')
    else:
        print(f'\n— PM median R²={pm_median:+.3f}, did not cross -1.0 threshold')


## 5e. Train M1 + Axis 2 — fast path (preload-to-GPU + bf16)

Adds three things on top of M1:
1. Edge attention with sparsity penalty (λ=1e-3)
2. Counterfactual edge dropout (p=0.07)
3. Multi-step rollout backprop, k-curriculum 1→2→4→4

**Performance changes from the streaming variant** (which was
running at 186 s/s):
- Preload all 49 train + 1 val replicates to GPU once. ~6 GB at bf16.
  No DataLoader, no parquet on hot path, no num_workers.
- Index iterator on GPU; each batch is a fancy-index gather.
- bf16 autocast (Blackwell tensor cores).
- `use_checkpoint=False` (96 GB VRAM headroom; no recompute).
- On-GPU loss accumulators; one CUDA sync per `log_every` interval.

Expected throughput: **5,000-15,000 samples/sec** (~30-80× the streaming
path). Full 4-epoch run: ~10-20 min on RTX PRO 6000 Blackwell.


In [ ]:
import importlib
import cell_sim.lgnn.models.gnn_v1_axis2 as _m
import cell_sim.lgnn.training.train_m1_axis2_fast as _t
importlib.reload(_m); importlib.reload(_t)

from cell_sim.lgnn.training.train_m1_axis2_fast import (
    M1Axis2FastTrainConfig, train_m1_axis2_fast)
from cell_sim.lgnn.data.species_graph import load_species_graph

g_full = load_species_graph(graph_path)
assert g_full.n_nodes == n_species

# Axis-2 round 2: attention warmup, lower λ_attn peak (1e-4), lower
# λ_dropout (0.1), skip rollout at k=1. Round 1 collapsed attention
# onto self-loops because the entropy penalty fired before the model
# learned which edges matter.
cfg_m1a2 = M1Axis2FastTrainConfig(
    n_species=n_species,
    hidden=64, n_layers=3,
    batch_size=128,
    edge_chunk_size=None,
    lr=3e-4, weight_decay=1e-5,
    train_replicates=tuple(range(1, 50)),
    val_replicates=(50,),
    n_epochs=4,
    device='auto', seed=42, log_every=50,
    edge_dropout_p=0.07,
    lambda_dropout=0.1,             # was 0.5 — reviewer fix #4
    lambda_attn=1e-4,               # peak; was 1e-3 — reviewer fix #2
    lambda_attn_warmup_steps=2000,  # no penalty for first 2000 steps — fix #1
    lambda_attn_ramp_steps=2000,    # ramp linearly over next 2000 — fix #1
    skip_rollout_at_k1=True,        # fix #5: don't double-count single-step
    k_curriculum=(1, 2, 4, 4),
    rollout_gamma=0.95, lambda_rollout=1.0,
    max_k=4,
    use_checkpoint=False, use_bf16=True,
    preload_dtype='bfloat16',
    wall_clock_budget_s=2 * 3600,
)
ckpt_m1a2 = CKPT_DIR / 'count_dynamics_gnn_v1_axis2_v2.pt'
out_m1a2 = train_m1_axis2_fast(cfg_m1a2, lsdata, g_full,
                                checkpoint_path=ckpt_m1a2)
print('\n=== M1+Axis2 v2 training complete ===')
print(f'best val MSE (single-step) : {out_m1a2["best_val_singlestep_mse"]:.4f}')
print(f'wall clock                 : {out_m1a2["wall_clock_total_s"]/3600:.2f} h')
print(f'k_per_epoch                : {out_m1a2["history"]["k_per_epoch"]}')
print(f'samples/sec per epoch      : {out_m1a2["history"]["samples_per_sec"]}')
print(f'λ_attn end of each epoch   : {out_m1a2["history"]["lambda_attn_end_of_epoch"]}')
print(f'train_full_ss per epoch    : {out_m1a2["history"]["train_full_ss"]}')
print(f'val_singlestep_mse per ep  : {out_m1a2["history"]["val_singlestep_mse"]}')
print(f'checkpoint                 : {ckpt_m1a2}')


In [ ]:
# === M1+Axis2 headlines + memorization metrics + edge-attribution sanity ===
import numpy as np, re, torch
from cell_sim.lgnn.models.gnn_v1_axis2 import CellGNNv1Axis2
from cell_sim.lgnn.data.dataset import replicate_to_log1p_array
from cell_sim.lgnn.training.train_m1_axis2 import evaluate_memorization
from cell_sim.lgnn.data.species_graph import EdgeKind

ckpt = torch.load(ckpt_m1a2, map_location='cpu', weights_only=False)
model_m1a2 = CellGNNv1Axis2(graph=g_full, hidden=cfg_m1a2.hidden,
                            n_layers=cfg_m1a2.n_layers, use_checkpoint=False)
model_m1a2.load_state_dict(ckpt['state_dict'])
model_m1a2.eval()

df = lsdata.load_replicate(50)
X = replicate_to_log1p_array(df)
DX = X[1:] - X[:-1]
X_in = X[:-1]
with torch.no_grad():
    P = []
    for s in range(0, X_in.shape[0], 128):
        P.append(model_m1a2(torch.from_numpy(X_in[s:s+128])).numpy())
PRED = np.concatenate(P, axis=0)

ss_res = ((PRED - DX) ** 2).sum(axis=0)
ss_tot = ((DX - DX.mean(axis=0, keepdims=True)) ** 2).sum(axis=0)
with np.errstate(divide='ignore', invalid='ignore'):
    r2 = 1.0 - ss_res / np.where(ss_tot > 0, ss_tot, 1.0)
r2 = np.where(ss_tot > 0, r2, np.nan)
names = list(df.index)
var_per_row = X.var(axis=0)

# Headlines A and B
top_idx = np.argsort(var_per_row)[::-1][:100]
print('=== M1+axis2 headline A: top-100 high-variance ===')
print(f'  median R² : {np.nanmedian(r2[top_idx]):+.3f}'
      f'   (M0a {-0.502:+.3f},  M1 {-0.005:+.3f})')
print(f'  >0.0      : {(r2[top_idx] > 0.0).sum()} / 100')

pm_idx = np.array([i for i, n in enumerate(names)
                   if re.match(r'^PM_\d{3,}', n)], dtype=np.int64)
if len(pm_idx) > 0:
    pm_r2 = r2[pm_idx]
    pm_finite = pm_r2[np.isfinite(pm_r2)]
    print(f'\n=== M1+axis2 headline B: PM_xxxx rows (the thesis test) ===')
    print(f'  count       : {len(pm_idx)} ({np.isfinite(pm_r2).sum()} finite)')
    print(f'  median R²   : {np.nanmedian(pm_r2):+.3f}'
          f'   (M0 {-1.494:+.3f}, M1 {-0.040:+.3f})')
    print(f'  >0.0        : {(pm_finite > 0).sum()} / {len(pm_finite)}'
          f'   (M0: 0 / 453, M1: 2 / 453)')
    pm_med = float(np.nanmedian(pm_r2))
    if pm_med > 0.0:
        print('\n*** AXIS-2 WIN: PM median R² > 0 ***')
    elif pm_med > -0.04:
        print('\n*** Axis-2 improvement over M1 baseline ***')
    else:
        print(f'\n— PM median R² {pm_med:+.3f}, no improvement over M1')

# k=10 rollout MSE bound check
with torch.no_grad():
    x_pred = torch.from_numpy(X[:1000]).clone()
    rolls = [x_pred.clone()]
    for _ in range(10):
        dx = model_m1a2(x_pred)
        x_pred = x_pred + dx
        rolls.append(x_pred.clone())
ss = (rolls[1] - torch.from_numpy(X[1:1001])).pow(2).mean().item()
rs = (rolls[10] - torch.from_numpy(X[10:1010])).pow(2).mean().item()
print(f'\n=== rollout stability check ===')
print(f'  single-step MSE          : {ss:.4f}')
print(f'  k=10 rollout MSE         : {rs:.4f}')
print(f'  ratio (target: < 4×)     : {rs/max(ss,1e-12):.2f}')

# Memorization detectors
print('\n=== memorization metrics (M2 spec preview, computed for M1+axis2) ===')
mem = evaluate_memorization(
    model_m1a2, lsdata, replicates=(48, 49, 50),
    device=torch.device('cpu'),
    split_time=3600,
)
print(f'  within-replicate mse ratio : '
      f'{mem["within_replicate_mse_ratio"]:.3f}'
      f'   (target < 1.5;  >3 = memorizing)')
print(f'  perturbation response ratios:')
for k_, v_ in mem['perturbation_response_ratios'].items():
    print(f'    {k_:12s} {v_:.4f}')

# Edge-attribution sanity: 5 hand-picked PM rows, top-attention incoming edge
# Since the layer's attention is internal, expose it via a one-off forward
# that returns per-edge alpha for inspection.
print('\n=== edge-attribution sanity (5 hand-picked PM rows) ===')
from cell_sim.lgnn.models.gnn_v1_axis2 import _segment_softmax
pm_picks = ['PM_0001', 'PM_0042', 'PM_0106', 'PM_0392', 'PM_0651']
name_to_idx = {n: i for i, n in enumerate(g_full.row_names)}
sample_X = torch.from_numpy(X[100:101])  # one timepoint
model_m1a2.eval()
h = model_m1a2.input_proj(sample_X.unsqueeze(-1))
with torch.no_grad():
    # Inspect layer 0's attention via the fused MLP path
    layer = model_m1a2.layers[0]
    edge_index = model_m1a2.edge_index
    edge_attr  = model_m1a2.edge_attr
    edge_kind  = model_m1a2.edge_kind
    src_all = edge_index[0]
    dst_all = edge_index[1]
    h_src = h.index_select(1, src_all)
    h_dst = h.index_select(1, dst_all)
    attr_b = edge_attr.unsqueeze(0).expand(1, -1, -1)
    kind_emb = layer.kind_embedding(edge_kind)
    kind_emb_b = kind_emb.unsqueeze(0).expand(1, -1, -1)
    inp = torch.cat([h_src, h_dst, attr_b, kind_emb_b], dim=-1)
    logits = layer.attn_mlp(inp).squeeze(-1)              # (1, E)
    alpha = _segment_softmax(logits, dst_all, g_full.n_nodes)  # (1, E)
    src_sorted = src_all
    dst_cat = dst_all

n_correct_rp = 0
for pm_name in pm_picks:
    if pm_name not in name_to_idx:
        print(f'  {pm_name:12s} not in graph')
        continue
    pm_i = name_to_idx[pm_name]
    locus = pm_name.split('_')[1]
    inc_mask = (dst_cat == pm_i)
    if not inc_mask.any():
        print(f'  {pm_name:12s}: no incoming edges')
        continue
    edge_pos = inc_mask.nonzero(as_tuple=True)[0]
    inc_alpha = alpha[0, edge_pos]
    inc_src = src_sorted[edge_pos]
    top_idx_local = int(inc_alpha.argmax().item())
    top_src_idx = int(inc_src[top_idx_local].item())
    top_src_name = g_full.row_names[top_src_idx]
    is_rp_match = top_src_name == f'RP_{locus}' or top_src_name == f'RPM_{locus}'
    n_correct_rp += int(is_rp_match)
    print(f'  {pm_name:12s}  top_src={top_src_name:12s}'
          f'  α={inc_alpha[top_idx_local].item():.3f}'
          f'   {"OK" if is_rp_match else ""}')
print(f'\n  RP/RPM as top attention for {n_correct_rp}/5 PM rows'
      f'   (acceptance: ≥4/5)')


## 6. Inspect per-species R² + plot a few trajectories

Mean R² over 8572 rows is hard to interpret because most species are constant or sparse. Look at the top-100 highest-variance rows (where the model has any chance of explaining variance) and plot a couple of trajectories with the predicted `dx` rolled forward from x at t=0.

## 5f. Train M2 Phase 1 — CfC node dynamics

Replaces the per-layer LayerNorm-residual update with the closed-form
continuous-time variant from the M2 spec:

    W, b ← MLPs(agg + h_self)            # gating from messages
    W ← clamp(W, ±1/τ_min)              # τ_min = 0.1·Δt
    h_new = σ(-W·h - b) ⊙ A + σ(W·h + b) ⊙ B

where A, B are per-species learned biases. The CfC is the closed-form
solution to a leaky-integrator ODE with input-dependent decay — fast
metabolites get short effective τ, slow proteins get long τ. Directly
addresses the M1+axis2 ceiling at single-step prediction.

Phase 1 keeps everything else the same as M1+axis2: edge attention,
counterfactual dropout, multi-step rollout, λ_attn / λ_dropout warmup.
Phase 2 will add the information bottleneck + perturbation augmentation
if Phase 1 confirms CfC is the right lever.

**Param count:** ~3.4M (vs M1+axis2's 69K) — the per-species A, B
biases dominate. Smaller default batch size (64 → 5.75 GB preload
+ ~12 GB activations) keeps memory comfortable.


In [ ]:
import importlib
import cell_sim.lgnn.models.gnn_v2 as _m2
import cell_sim.lgnn.training.train_m2 as _t2
importlib.reload(_m2); importlib.reload(_t2)

from cell_sim.lgnn.training.train_m2 import M2TrainConfig, train_m2
from cell_sim.lgnn.data.species_graph import load_species_graph

g_full = load_species_graph(graph_path)
assert g_full.n_nodes == n_species

cfg_m2 = M2TrainConfig(
    n_species=n_species,
    hidden=64, n_layers=3,
    batch_size=64,                  # halved from M1+axis2 — CfC adds ~3M params
    edge_chunk_size=None,
    lr=3e-4, weight_decay=1e-5,
    train_replicates=tuple(range(1, 50)),
    val_replicates=(50,),
    n_epochs=4,
    device='auto', seed=42, log_every=50,
    edge_dropout_p=0.07,
    lambda_dropout=0.1,
    lambda_dropout_warmup_steps=2000,
    lambda_dropout_ramp_steps=2000,
    lambda_attn=1e-4,
    lambda_attn_warmup_steps=2000,
    lambda_attn_ramp_steps=2000,
    skip_rollout_at_k1=True,
    k_curriculum=(1, 2, 4, 4),
    rollout_gamma=0.95, lambda_rollout=1.0,
    max_k=4,
    use_checkpoint=False,
    use_bf16=True,
    preload_dtype='bfloat16',
    cfc_tau_min=0.1,                # τ_min in units of Δt (=1s)
    wall_clock_budget_s=3 * 3600,
)
ckpt_m2 = CKPT_DIR / 'count_dynamics_gnn_v2_p1.pt'
out_m2 = train_m2(cfg_m2, lsdata, g_full, checkpoint_path=ckpt_m2)
print('\n=== M2 Phase 1 (CfC) training complete ===')
print(f'best val MSE (single-step) : {out_m2["best_val_singlestep_mse"]:.4f}')
print(f'wall clock                 : {out_m2["wall_clock_total_s"]/3600:.2f} h')
print(f'k_per_epoch                : {out_m2["history"]["k_per_epoch"]}')
print(f'samples/sec per epoch      : {out_m2["history"]["samples_per_sec"]}')
print(f'val_singlestep_mse / ep    : {out_m2["history"]["val_singlestep_mse"]}')
print(f'val_mse_per_step / ep      : {out_m2["history"]["val_mse_per_step"]}')
print(f'val_rollout_mse_avg / ep   : {out_m2["history"]["val_rollout_mse_avg"]}')
print(f'checkpoint                 : {ckpt_m2}')


In [ ]:
# === M2 eval — same headlines as M1+axis2, comparable directly ===
import numpy as np, re, torch
from cell_sim.lgnn.models.gnn_v2 import CellGNNv2
from cell_sim.lgnn.data.dataset import replicate_to_log1p_array

device = torch.device('cuda')
ckpt = torch.load(ckpt_m2, map_location='cpu', weights_only=False)
state = {k: v.to(torch.float32) for k, v in ckpt['state_dict'].items()}
model_m2 = CellGNNv2(graph=g_full, hidden=64, n_layers=3,
                     use_checkpoint=False, cfc_tau_min=0.1).to(device)
model_m2.load_state_dict(state)
model_m2.eval()

df = lsdata.load_replicate(50)
X = replicate_to_log1p_array(df)
DX = X[1:] - X[:-1]
X_in = X[:-1]
names = list(df.index)
with torch.no_grad():
    P = []
    for s in range(0, X_in.shape[0], 128):
        xb = torch.from_numpy(X_in[s:s+128]).to(device)
        P.append(model_m2(xb).cpu().numpy())
PRED = np.concatenate(P, axis=0)

ss_res = ((PRED - DX) ** 2).sum(axis=0)
ss_tot = ((DX - DX.mean(axis=0, keepdims=True)) ** 2).sum(axis=0)
with np.errstate(divide='ignore', invalid='ignore'):
    r2 = 1.0 - ss_res / np.where(ss_tot > 0, ss_tot, 1.0)
r2 = np.where(ss_tot > 0, r2, np.nan)
var_per_row = X.var(axis=0)

top_idx = np.argsort(var_per_row)[::-1][:100]
print('=== M2 headline A: top-100 high-variance ===')
print(f'  median R² : {np.nanmedian(r2[top_idx]):+.3f}'
      f'   (M0a {-0.502:+.3f}, M1 {-0.005:+.3f}, axis2-v3 {-0.368:+.3f})')
print(f'  >0.0      : {(r2[top_idx] > 0.0).sum()} / 100')

pm_idx = np.array([i for i, n in enumerate(names)
                   if re.match(r'^PM_\d{3,}', n)], dtype=np.int64)
pm_r2 = r2[pm_idx]
pm_finite = pm_r2[np.isfinite(pm_r2)]
print(f'\n=== M2 headline B: PM_xxxx (the M2 thesis test) ===')
print(f'  count       : {len(pm_idx)} ({np.isfinite(pm_r2).sum()} finite)')
print(f'  median R²   : {np.nanmedian(pm_r2):+.3f}'
      f'   (M0 {-1.494:+.3f}, M1 {-0.040:+.3f}, axis2-v3 {-0.331:+.3f})')
print(f'  10th pct R² : {np.nanpercentile(pm_r2, 10):+.3f}'
      f'   (M0 {-4.572:+.3f}, M1 {-0.094:+.3f}, axis2-v3 {-0.844:+.3f})')
print(f'  >0.0        : {(pm_finite > 0).sum()} / {len(pm_finite)}'
      f'   (M0: 0/453, M1: 2/453, axis2-v3: 0/453)')

pm_med = float(np.nanmedian(pm_r2))
if pm_med > 0.1:
    print('\n*** M2 ACCEPTANCE: PM median R² > 0.1 ***')
elif pm_med > -0.04:
    print('\n*** M2 PARITY OR BETTER vs M1 baseline ***')
else:
    print(f'\n— PM median R² {pm_med:+.3f}, did not beat M1 baseline')


In [ ]:
import numpy as np
import re
import torch
from cell_sim.lgnn.models.mlp_baseline import CountDynamicsMLP
from cell_sim.lgnn.data.dataset import replicate_to_log1p_array

ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
model = CountDynamicsMLP(
    n_species=cfg.n_species, hidden=cfg.hidden, n_blocks=cfg.n_blocks)
model.load_state_dict(ckpt['state_dict'])
model.eval()

df = lsdata.load_replicate(50)                       # held-out
X = replicate_to_log1p_array(df)                     # (T, S) signed-log1p
T = X.shape[0]
DX = X[1:] - X[:-1]                                  # (T-1, S) ground-truth dx

X_in = X[:-1]
with torch.no_grad():
    pred = []
    bs = 512
    for s in range(0, X_in.shape[0], bs):
        x = torch.from_numpy(X_in[s:s+bs])
        pred.append(model(x).numpy())
PRED = np.concatenate(pred, axis=0)                  # (T-1, S)
assert PRED.shape == DX.shape, (PRED.shape, DX.shape)

# Per-species R²
ss_res = ((PRED - DX) ** 2).sum(axis=0)
ss_tot = ((DX - DX.mean(axis=0, keepdims=True)) ** 2).sum(axis=0)
with np.errstate(divide='ignore', invalid='ignore'):
    r2 = 1.0 - ss_res / np.where(ss_tot > 0, ss_tot, 1.0)
r2 = np.where(ss_tot > 0, r2, np.nan)

names = list(df.index)
var_per_row = X.var(axis=0)

# === Headline metric A: top-100 high-variance rows (the M0-era headline) ===
top_idx = np.argsort(var_per_row)[::-1][:100]
print('=== headline A: top-100 high-variance rows ===')
print(f'  median R² : {np.nanmedian(r2[top_idx]):+.3f}')
print(f'  mean R²   : {np.nanmean(r2[top_idx]):+.3f}')
print(f'  >0.5      : {(r2[top_idx] > 0.5).sum()} / 100')
print(f'  >0.0      : {(r2[top_idx] > 0.0).sum()} / 100')

# === Headline metric B: PM_xxxx rows specifically (the M1 thesis test) ===
# This is the metric the graph build was designed to move. M0 hits ~-8 here.
# M1 should ideally beat -1 (graph thesis) and ideally cross 0 (graph wins).
pm_idx = np.array([i for i, n in enumerate(names) if re.match(r'^PM_\d{3,}', n)],
                  dtype=np.int64)
if len(pm_idx) > 0:
    pm_r2 = r2[pm_idx]
    pm_r2_finite = pm_r2[np.isfinite(pm_r2)]
    print(f'\n=== headline B: PM_xxxx rows (M1 thesis target) ===')
    print(f'  count        : {len(pm_idx)} ({np.isfinite(pm_r2).sum()} finite)')
    print(f'  median R²    : {np.nanmedian(pm_r2):+.3f}')
    print(f'  10th pct R²  : {np.nanpercentile(pm_r2, 10):+.3f}')
    print(f'  >0.0         : {(pm_r2_finite > 0).sum()} / {len(pm_r2_finite)}')
    print(f'  > -1.0       : {(pm_r2_finite > -1.0).sum()} / {len(pm_r2_finite)}')
else:
    print('\n=== headline B: no PM_xxxx rows in this dataset ===')

# Top/bottom-fit examples (high-variance subset)
ranked = sorted(top_idx, key=lambda i: -r2[i] if np.isfinite(r2[i]) else 1e9)
print(f'\n5 best-fit high-variance rows:')
for i in ranked[:5]:
    print(f'  {names[i]:30s}  R²={r2[i]:+.3f}  var={var_per_row[i]:.3f}')
print(f'\n5 worst-fit high-variance rows:')
for i in ranked[-5:]:
    print(f'  {names[i]:30s}  R²={r2[i]:+.3f}  var={var_per_row[i]:.3f}')


In [ ]:
import matplotlib.pyplot as plt

# Roll out the predicted dx from x at t=0 and compare to ground truth
x = X[0].copy()
rollout = [x.copy()]
with torch.no_grad():
    for _ in range(T - 1):
        dx = model(torch.from_numpy(x).unsqueeze(0)).squeeze(0).numpy()
        x = x + dx
        rollout.append(x.copy())
ROLL = np.stack(rollout, axis=0)                     # (T, S) in log1p space

# Pick 4 representative high-variance rows
rows_to_plot = ranked[:4]
fig, axs = plt.subplots(2, 2, figsize=(12, 7))
for ax, idx in zip(axs.flat, rows_to_plot):
    t = np.arange(T)
    ax.plot(t, X[:, idx],     label='truth (signed-log1p)',       lw=1)
    ax.plot(t, ROLL[:, idx],  label='rollout',                     lw=1)
    ax.set_title(f'{names[idx]}   R²={r2[idx]:+.3f}', fontsize=10)
    ax.set_xlabel('time (s)')
    ax.legend(fontsize=8)
fig.suptitle('Replicate 50 hold-out: ground truth vs autoregressive rollout',
             y=1.02)
fig.tight_layout()
plt.show()

### Reading the results

- **Median R² over top-100 high-variance rows** is the headline single number. Above 0.5 = the surrogate explains a meaningful chunk of the dynamics on the species that actually move; near 0 = the model is fitting a constant-mean baseline; sharply negative = autoregressive instability or a transform bug.
- **Rollout drift**: the rollout plot is the harder test — even small per-step errors accumulate. If the trained model can't roll out for 7200 steps without going off-manifold, that's expected for a v0 single-step MLP and motivates a v2 with teacher-forcing + a longer-horizon loss.
- **Worst-fit rows** are usually species whose dynamics are dominated by stochastic firing of low-rate reactions; a deterministic mean predictor can't follow them. That's a property of the data, not the model.